# Étape 5 - Prévision et visualisation

On utilise le modèle entraîné pour prédire le chiffre d'affaires de la semaine à venir,
puis on trace le résultat.

## 1. Importer les librairies

In [1]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:156: FutureWarning: Model's `predict` method contains invalid parameters: {'X'}. Only the following parameter names are allowed: context, model_input, and params. Note that invalid parameters will no longer be permitted in future versions.
  param_names = _check_func_signature(func, "predict")


************************************************************
USING default value : foodcast.settings.dev
************************************************************


## 2. Reprendre les étapes 1 à 4

In [2]:
# --- Reprise des étapes 1 à 4 ---
# jeu d'entraînement
df = etl(settings.DATA_DIR, 197, 200)
df = features_offline(df)
x_train = df.drop(columns=['cash_in']).set_index('order_date')
y_train = df[['order_date', 'cash_in']].set_index('order_date')['cash_in']

# jeu de prédiction
past = etl(settings.DATA_DIR, 200, 200)
future = span_future(past['order_date'].max())
future = features_online(future, past)
future = future.set_index('order_date')

# modèle simple entraîné sur tout
simple_model = RandomForestRegressor(n_estimators=10, random_state=42)
simple_model.fit(x_train, y_train)
future.head()

2026-09-09 15:30:16 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (2158, 6)
2026-09-09 15:30:16 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (3247, 6)
2026-09-09 15:30:16 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (380, 3)
2026-09-09 15:30:16 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (565, 3)
2026-09-09 15:30:16 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - merge: shape = (945, 2)
2026-09-09 15:30:16 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - resample: shape = (659, 2)
2026-09-09 15:30:16 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - IN

,day_1,day_2,day_3,day_4,day_5,day_6,hour_cos_1,hour_sin_1,lag_1W
order_date,,,,,,,,,
2018-11-05 00:00:00,False,False,False,False,False,False,1.000000,0.000000,0.0
2018-11-05 01:00:00,False,False,False,False,False,False,0.965926,0.258819,0.0
2018-11-05 02:00:00,False,False,False,False,False,False,0.866025,0.500000,0.0
2018-11-05 03:00:00,False,False,False,False,False,False,0.707107,0.707107,0.0
2018-11-05 04:00:00,False,False,False,False,False,False,0.500000,0.866025,0.0


## 3. Prédire le chiffre d'affaires futur

`simple_model.predict(future)` renvoie un tableau numpy. On le range dans un `DataFrame`
avec **le même index que `future`** et une seule colonne `y_pred_simple`
(ce nom est celui attendu par `plotly_predictions`).

In [3]:
y_pred = pd.DataFrame(
    simple_model.predict(future),
    index=future.index,
    columns=['y_pred_simple'],
)
y_pred.head(20)

,y_pred_simple
order_date,
2018-11-05 00:00:00,0.000
2018-11-05 01:00:00,0.000
2018-11-05 02:00:00,0.000
2018-11-05 03:00:00,0.000
2018-11-05 04:00:00,0.000
2018-11-05 05:00:00,0.000
2018-11-05 06:00:00,0.000
2018-11-05 07:00:00,0.000
2018-11-05 08:00:00,0.000


## 4. Tracer la prévision

In [4]:
plotly_predictions(y_pred)

2026-09-09 15:30:23 - foodcast.domain.forecast - INFO - plotly_predictions: predictions shape = (168, 1)


On obtient **une seule courbe** de prévision. Elle reproduit bien les cycles
jour/nuit et semaine, mais ne dit rien sur la **confiance** qu'on peut lui accorder.

➡️ Étape suivante : `06_incertitudes_multimodel.ipynb`